# Practical Core — 6 Indicators

| Aspect                          | Includes                                                                      |
| ------------------------------- | ----------------------------------------------------------------------------- |
| Monetary & Financial Conditions | Interest rates, central banks, liquidity, credit, stock/bond markets, banking |
| Inflation & Prices              | CPI, PPI, wages, commodity-driven price pressures                             |
| Real Economic Activity          | GDP, recession, manufacturing, services, investment, productivity             |
| Labor & Consumption             | Employment, wages, retail sales, consumer spending/confidence                 |
| Fiscal & Government             | Taxes, deficits, spending, regulation, stimulus                               |
| External Sector                 | Trade, exports/imports, FX, tariffs, current account                          |



In [1]:
from google.colab import userdata

token = userdata.get("Github_Token")
!git clone https://{token}@github.com/Nas-Mohd/economic-news-sentiment.git
!git config --global user.email "anasamohammad.school@gmail.com"
!git config --global user.name "Nas-Mohd"

fatal: destination path 'economic-news-sentiment' already exists and is not an empty directory.


In [2]:
!pip install snorkel # Run once


In [2]:

import sys
sys.path.append('/content/economic-news-sentiment')

In [3]:
!git -C /content/economic-news-sentiment/ pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.58 KiB | 147.00 KiB/s, done.
From https://github.com/Nas-Mohd/economic-news-sentiment
   b2e7ad0..d4df00a  main       -> origin/main
Updating b2e7ad0..d4df00a
Fast-forward
 labeling/anchors.py | 47 ++++++++++++++++++++++++++++++++++++++++++++++-
 1 file changed, 46 insertions(+), 1 deletion(-)


In [ ]:
!git -C /content/economic-news-sentiment add .
!git -C /content/economic-news-sentiment commit -m "made semantic ABSENT threshold lower"
!git -C /content/economic-news-sentiment push

[main 084f4cd] made semantic ABSENT threshold lower
 6 files changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 2 threads
Compressing objects: 100% (12/12), done.
Writing objects: 100% (12/12), 13.33 KiB | 593.00 KiB/s, done.
Total 12 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/Nas-Mohd/economic-news-sentiment.git
   71c1eae..084f4cd  main -> main


In [4]:
# RUN EVERYTIME CHANGES ARE MADE
import importlib
import labeling
from labeling.labeling_functions import make_keyword_lf, make_semantic_lf
from labeling.labeling_functions.lf import embedder
from labeling.anchors import ANCHORS, SEMANTIC_THRESHOLD
from labeling.keywords import KEYWORDS
from labeling.labeling_functions.lf_factories import nlp
from labeling.labeling_functions import lf_lemmatized_cost_push, lf_is_not_economic_news, lf_monetary_policy_inflation_exclusion, lf_central_banks_nlp, lf_rate_actions_nlp, lf_labor_market_action_nlp, lf_central_bank_fiscal_exclusion, lf_fiscal_sector_clash_exclusion, lf_monetary_exclusion_clash
from labeling.labeling_functions import lf_inflation_exclusion_clash, lf_activity_exclusion_clash, lf_labor_exclusion_clash, lf_fiscal_exclusion_clash, lf_external_exclusion_clash, lf_deny_fiscal_if_pure_labor, lf_advanced_context_regex_trade, lf_advanced_context_regex_real, lf_fiscal_policy_semantic_false, lf_fiscal_policy_semantic_true
from labeling.labeling_functions import lf_fiscal_actions_nlp
importlib.reload(labeling)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

<module 'labeling' (namespace) from ['/content/economic-news-sentiment/labeling']>

In [5]:
import re
from snorkel.preprocess import preprocessor
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.lf.nlp import nlp_labeling_function
from snorkel.labeling.model import LabelModel
from sentence_transformers import SentenceTransformer, util
from typing import List, Dict
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
import numpy as np

In [6]:
SEMANTIC_THRESHOLD

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [7]:
ECONOMIC_ASPECTS = ["monetary_policy", "inflation", "economic_growth", "labor_market", "consumer_activity", "business_activity", "financial_markets", "trade_external", "fiscal_policy", "energy_commodities", "banking_credit", "corporate_climate"]

# --- Convert to dict of list of tensors ---
ANCHORS_EMBEDDING: Dict[str, List] = {}  # Value type will be list[tensor]
for category in ECONOMIC_ASPECTS:
    ANCHORS_EMBEDDING[category] = [
        embedder.encode(phrase, convert_to_tensor=True) for phrase in ANCHORS[category]
    ]


In [8]:
# Instantiating the merged, simplified keyword and semantic similarity LFs
lf_monetary_policy_keywords = make_keyword_lf("monetary_policy", KEYWORDS)
lf_monetary_policy_semantic = make_semantic_lf("monetary_policy", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_inflation_keywords = make_keyword_lf("inflation", KEYWORDS)
lf_inflation_semantic = make_semantic_lf("inflation", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_economic_growth_keywords = make_keyword_lf("economic_growth", KEYWORDS)
lf_economic_growth_semantic = make_semantic_lf("economic_growth", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_labor_market_keywords = make_keyword_lf("labor_market", KEYWORDS)
lf_labor_market_semantic = make_semantic_lf("labor_market", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_consumer_activity_keywords = make_keyword_lf("consumer_activity", KEYWORDS)
lf_consumer_activity_semantic = make_semantic_lf("consumer_activity", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_business_activity_keywords = make_keyword_lf("business_activity", KEYWORDS)
lf_business_activity_semantic = make_semantic_lf("business_activity", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_financial_markets_keywords = make_keyword_lf("financial_markets", KEYWORDS)
lf_financial_markets_semantic = make_semantic_lf("financial_markets", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_trade_external_keywords = make_keyword_lf("trade_external", KEYWORDS)
lf_trade_external_semantic = make_semantic_lf("trade_external", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_fiscal_policy_keywords = make_keyword_lf("fiscal_policy", KEYWORDS)
lf_fiscal_policy_semantic = make_semantic_lf("fiscal_policy", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_energy_commodities_keywords = make_keyword_lf("energy_commodities", KEYWORDS)
lf_energy_commodities_semantic = make_semantic_lf("energy_commodities", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_banking_credit_keywords = make_keyword_lf("banking_credit", KEYWORDS)
lf_banking_credit_semantic = make_semantic_lf("banking_credit", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_central_banks_keywords = make_keyword_lf("central_banks", KEYWORDS)
lf_rate_actions = make_keyword_lf("rate_actions", KEYWORDS)

In [9]:
# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab', quiet=True)
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [10]:
# Embedder model instantiation
#embedder = SentenceTransformer('all-mpnet-base-v2')
#embedder = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')
#embedder = SentenceTransformer('sentence-transformers/all-distilroberta-v1')

In [10]:
import json

# Load JSON file
with open("/content/economic-news-sentiment/labeling/data/predictions_review.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Flatten into rows
rows = []

for aspect_combo, samples in data.items():
    for sample in samples:
        rows.append({
            "aspects": aspect_combo,
            "article_id": sample.get("article_id"),
            "title": sample.get("title"),
            "sentence_idx": sample.get("sentence_idx"),
            "text": sample.get("text")
        })

# Create DataFrame
df = pd.DataFrame(rows)

# Optional: inspect
print(df.head())

                               aspects  article_id  \
0                                 None         164   
1                                 None           3   
2                      External_Sector          97   
3                      External_Sector          65   
4  External_Sector + Fiscal_Government         159   

                                               title  sentence_idx  \
0         Where experience meets academic excellence            14   
1  ECB urges banks to brace quickly for AI-assist...             8   
2  China's durian boom is ripening into a regiona...            21   
3  Johor eyeing 12 million visitors, RM42.48bil i...             7   
4              Cautious optimism on growth prospects             4   

                                                text  
0  “I needed a university that was recognised, ap...  
1  (Reporting by Ludwig Burger and Reinhard Becke...  
2  As Southeast Asia's main producing areas enter...  
3  He added that Singapore remains

In [11]:
# When extracting your final training labels from the matrix:
def filter_high_confidence_solo_votes(label_matrix, probabilities, threshold=0.5):
    final_labels = []

    for row_votes, prob in zip(label_matrix, probabilities):
        # Count how many active (non-abstain) votes are present
        # (Assuming ABSTAIN is coded as -1 or 0 depending on your setup)
        active_votes = sum(1 for vote in row_votes if vote != -1)

        # Guardrail: If only 1 LF voted, don't trust a runaway high probability
        if active_votes == 1 and prob > 0.90:
            final_labels.append(-1) # Force to ABSTAIN/Ignore for training
        else:
            final_labels.append(1 if prob >= threshold else 0)

    return final_labels

In [12]:
ASPECTS = [ # 6 MAIN ASPECTS TO LOOK FOR IN TEXT
    "Monetary_Financial",
    "Inflation_Prices",
    "Real_Economic_Activity",
    "Labor_Consumption",
    "Fiscal_Government",
    "External_Sector"
]

monetary_lfs = [
    lf_is_not_economic_news,
    lf_monetary_policy_keywords,
    lf_monetary_policy_semantic,
    lf_financial_markets_keywords,
    lf_financial_markets_semantic,
    lf_banking_credit_keywords,
    lf_banking_credit_semantic,
    lf_central_banks_keywords,
    lf_central_banks_nlp,
    lf_rate_actions_nlp,
    lf_monetary_policy_inflation_exclusion,
    lf_monetary_exclusion_clash
]

inflation_lfs = [
    lf_is_not_economic_news,
    lf_inflation_keywords,
    lf_inflation_semantic,
    lf_monetary_policy_inflation_exclusion,
    lf_rate_actions_nlp,
    lf_inflation_exclusion_clash,
    lf_lemmatized_cost_push
]

activity_lfs = [
    lf_is_not_economic_news,
    lf_economic_growth_keywords,
    lf_economic_growth_semantic,
    lf_business_activity_keywords,
    lf_business_activity_semantic,
    lf_trade_external_keywords,
    lf_activity_exclusion_clash,
    lf_advanced_context_regex_real  # Highly critical here to catch physical "supply shocks"
]

labor_lfs = [
    lf_is_not_economic_news,
    lf_labor_market_keywords,
    lf_labor_market_semantic,
    lf_labor_exclusion_clash,
    lf_consumer_activity_keywords,
    lf_consumer_activity_semantic,
    lf_labor_market_action_nlp    # Catches structural labor-actions via dependency parsing
]

fiscal_lfs = [
    lf_is_not_economic_news,
    lf_fiscal_policy_keywords,
    lf_fiscal_policy_semantic,
    lf_central_bank_fiscal_exclusion,
    lf_deny_fiscal_if_pure_labor,
    lf_fiscal_actions_nlp
]

external_lfs = [
    lf_is_not_economic_news,
    lf_trade_external_keywords,
    lf_trade_external_semantic,
    lf_energy_commodities_keywords,
    lf_energy_commodities_semantic,
    lf_external_exclusion_clash,
    lf_advanced_context_regex_trade  # Highly critical here to catch "global oil prices" and geopolitical boundaries
]

# 2. Final Aspect Mapping Dictionary
aspect_lf_groups = {
    "Monetary_Financial": monetary_lfs,
    "Inflation_Prices": inflation_lfs,
    "Real_Economic_Activity": activity_lfs,
    "Labor_Consumption": labor_lfs,
    "Fiscal_Government": fiscal_lfs,
    "External_Sector": external_lfs
}

In [17]:
#── 1. Load the CSV ──────────────────────────────────────────────────────────
df = pd.read_csv("/content/economic-news-sentiment/webscraping/malaysian_economic_news_1000.csv")   # ← update path as needed


In [13]:
# ── 2. Explode each article into one row per sentence ────────────────────────
def explode_to_sentences(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for article_id, row in df.iterrows():
        title = row.get("title", "")
        text  = row.get("text", "")

        # headline as its own sentence row
        if pd.notna(title) and str(title).strip():
            rows.append({
                "article_id":   article_id,
                "title":        title,
                "sentence_idx": -1,           # -1 flags it as the headline
                "text":         str(title).strip(),
                "is_headline":  True,
            })

        # body sentences
        if pd.notna(text) and str(text).strip():
            sentences = sent_tokenize(str(text))
            for i, sent in enumerate(sentences):
                rows.append({
                    "article_id":   article_id,
                    "title":        title,
                    "sentence_idx": i,
                    "text":         sent.strip(),
                    "is_headline":  False,
                })

    return pd.DataFrame(rows)

In [19]:
sentence_df = explode_to_sentences(df)
print(f"\nTotal sentences: {len(sentence_df)}")
print(f"Total articles:  {sentence_df['article_id'].nunique()}")


Total sentences: 3475
Total articles:  182


In [ ]:
sample_df = sentence_df.sample(n=500, random_state=42)

In [ ]:
sample_df.to_csv("sample_df.csv", index=False)

In [20]:
# We'll store the LF matrices for each aspect
L_matrices = {}

for aspect, lf_list in aspect_lf_groups.items():
  applier = PandasLFApplier(lf_list)
  L_train = applier.apply(sentence_df)
  L_matrices[aspect] = L_train

# Train a LabelModel per aspect
label_models = {}
preds_proba = {}  # shape: (num_samples, 1) probability of class 1


100%|██████████| 3475/3475 [04:01<00:00, 14.36it/s]


In [42]:
import numpy as np

for aspect, L in L_matrices.items():
    # 1. Initialize the LabelModel
    label_model = LabelModel(
        cardinality=2,
        verbose=True
    )

    # 2. Fit with a specified optimizer learning rate and regularization penalty
    label_model.fit(
    L,
    n_epochs=200,
    seed=42,
    lr=0.01,
    l2=0.1,
    class_balance = [0.6,0.4]
)

    label_models[aspect] = label_model

    # 3. Get raw continuous probability scores for the PRESENT class (col 1)
    raw_probs = label_model.predict_proba(L)[:, 1]

    # --------------------------------------------------------------------------
    # GUARDRAIL: Identify and filter out runaway high-confidence solo votes
    # --------------------------------------------------------------------------
    # Count how many labeling functions voted (anything that isn't an ABSTAIN / -1)
    active_vote_counts = (L != -1).sum(axis=1)

    # Create a mask for rows where exactly ONE labeling function voted
    is_solo_vote = (active_vote_counts == 1)

    # Create a mask for rows where the model generated a runaway high probability
    is_runaway_confidence = (raw_probs > 0.80)

    # Combine masks: True if it's a dangerous solo voter with runaway confidence
    should_filter = is_solo_vote & is_runaway_confidence

    # Apply guardrail: Force dangerous rows to 0.0 probability (or map to a specific fallback index)
    sanitized_probs = np.where(should_filter, 0.0, raw_probs)

    # Store the sanitized probabilities for your final downstream extraction
    preds_proba[aspect] = sanitized_probs

100%|██████████| 200/200 [00:00<00:00, 930.14epoch/s]


In [43]:
for aspect, lf_list in aspect_lf_groups.items():
  print(f'\nASPECT: {aspect}')
  print(LFAnalysis(L=L_matrices[aspect], lfs=lf_list).lf_summary())



ASPECT: Monetary_Financial
                                         j Polarity  Coverage  Overlaps  \
lf_is_not_economic_news                  0      [0]  0.142446  0.104460   
lf_monetary_policy_nlp_matcher           1      [1]  0.043453  0.042014   
lf_monetary_policy_semantic              2   [0, 1]  0.202590  0.164317   
lf_financial_markets_nlp_matcher         3      [1]  0.065036  0.054676   
lf_financial_markets_semantic            4   [0, 1]  0.253525  0.160288   
lf_banking_credit_nlp_matcher            5      [1]  0.064748  0.058993   
lf_banking_credit_semantic               6   [0, 1]  0.235108  0.146763   
lf_central_banks_nlp_matcher             7      [1]  0.030791  0.030504   
lf_central_banks_nlp                     8      [1]  0.017266  0.017266   
lf_rate_actions_nlp                      9      [1]  0.001727  0.001151   
lf_monetary_policy_inflation_exclusion  10      [1]  0.019281  0.019281   
lf_monetary_exclusion_clash             11      [0]  0.009784  0.001439 

In [44]:
# Define the order you want (same as aspect_lf_groups)
aspect_order = [
    "Monetary_Financial",
    "Inflation_Prices",
    "Real_Economic_Activity",
    "Labor_Consumption",
    "Fiscal_Government",
    "External_Sector"
]

# Stack probabilities into an (n_samples, 6) array
probs_matrix = np.column_stack([preds_proba[aspect] for aspect in aspect_order])

# Or as a tidy DataFrame (each row is a sample, each column an aspect)
probs_df = pd.DataFrame(probs_matrix, columns=aspect_order)

# If you later need crisp 0/1 presence labels, threshold (e.g., at 0.5):
#presence_matrix = (probs_matrix >= 0.5).astype(int)
#presence_df = pd.DataFrame(presence_matrix, columns=aspect_order)

In [46]:

# ── 1. Concatenate ────────────────────────────────────────────────────────────
# probs_df columns are your label model probability outputs
#combined_df = pd.concat([sample_df.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)
combined_df = pd.concat([sentence_df.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)

print(combined_df.head())
print(f"\nShape: {combined_df.shape}")

# ── 2. Check for predictions significantly close to 0 or 1 ───────────────────
THRESHOLD = 0.4  # "close to 0" = < 0.15, "close to 1" = > 0.85

label_cols = probs_df.columns.tolist()

# Boolean mask: any label column is near 0 or 1
near_boundary = combined_df[label_cols].apply(
    lambda col: (col < THRESHOLD) | (col > (1 - THRESHOLD))
)

# Rows where AT LEAST ONE label is near a boundary
flagged = combined_df[near_boundary.any(axis=1)].copy()
flagged["flagged_labels"] = near_boundary[near_boundary.any(axis=1)].apply(
    lambda row: [label_cols[i] for i, v in enumerate(row) if v], axis=1
)

print(f"\nFlagged rows (any label near 0 or 1): {len(flagged)} / {len(combined_df)}")
print(flagged[["article_id", "text", "flagged_labels"] + label_cols].head(10))

# ── 3. Summary: per-label breakdown ──────────────────────────────────────────
print("\n── Per-label flagged counts ──")
for col in label_cols:
    n_near_zero = (combined_df[col] < THRESHOLD).sum()
    n_near_one  = (combined_df[col] > (1 - THRESHOLD)).sum()
    print(f"  {col:30s}  near 0: {n_near_zero:4d}  |  near 1: {n_near_one:4d}")

   article_id                                              title  \
0           0  Dialog records stronger 3Q26 as it keeps focus...   
1           0  Dialog records stronger 3Q26 as it keeps focus...   
2           0  Dialog records stronger 3Q26 as it keeps focus...   
3           0  Dialog records stronger 3Q26 as it keeps focus...   
4           0  Dialog records stronger 3Q26 as it keeps focus...   

   sentence_idx                                               text  \
0            -1  Dialog records stronger 3Q26 as it keeps focus...   
1             0  PETALING JAYA: Dialog Group Bhd is staying foc...   
2             1  The group said it will continue to build and s...   
3             2  Releasing its results for the third quarter (3...   
4             3  In a bourse filing, the group attributed the i...   

   is_headline  Monetary_Financial  Inflation_Prices  Real_Economic_Activity  \
0         True            0.000242          0.000337                0.000307   
1        F

In [47]:
# ── Filter by aspect ──────────────────────────────────────────────────────────
ASPECT = "Fiscal_Government"   # ← change this to any label col
DIRECTION = "near_1"            # ← "near_1", "near_0", or "both"

if DIRECTION == "near_1":
    mask = combined_df[ASPECT] > (1 - THRESHOLD)
elif DIRECTION == "near_0":
    mask = combined_df[ASPECT] < THRESHOLD
else:  # both
    mask = (combined_df[ASPECT] > (1 - THRESHOLD)) | (combined_df[ASPECT] < THRESHOLD)

aspect_flagged = combined_df[mask][["article_id", "text", ASPECT]]

print(f"── {ASPECT} | {DIRECTION} | {len(aspect_flagged)} rows ──")
for _, row in aspect_flagged.iterrows():
    print(f"\n [{row["article_id"]}] [{row[ASPECT]:.4f}]  {row['text'][:500]}")

── Fiscal_Government | near_1 | 358 rows ──

 [1] [0.6424]  "The minister is also expected to share Malaysia’s views on major global issues, including trade, global supply chains, energy security, digitalisation and climate action,” he said.

 [4] [0.6424]  “As a trading nation and a middle-class power, we really have an interest in diversifying our supply chains, forging new partnerships and investment opportunities.

 [5] [0.6424]  It noted improved earnings, growing supply chain dominance and a stronger yuan.

 [10] [0.6424]  Economic data showed consumer prices rose at a faster pace last month than analysts anticipated as the closure of the Strait of Hormuz due to the war with Iran continued to disrupt crude supply.

 [10] [0.6892]  "Warsh is not going to be able to cut rates even if he wants to, and I don't think he will want to," Hatfield said, adding he was optimistic about Warsh's Fed reform plans.

 [11] [0.6424]  PETALING JAYA: While food prices in Malaysia remain largely und

In [40]:
view_df = combined_df[combined_df["article_id"].isin([179])][['article_id',"text", "Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
    "Labor_Consumption", "Fiscal_Government", "External_Sector"]]

In [41]:
#view_df.iloc[1]
view_df.head(10)

,article_id,text,Monetary_Financial,Inflation_Prices,Real_Economic_Activity,Labor_Consumption,Fiscal_Government,External_Sector
3404,179,Malaysia urges US to reconsider move to block ...,2.225611e-04,0.400000,0.400000,0.400000,0.400000,5.684522e-01
3405,179,REMBAU: Malaysia has asked the United States (...,2.225611e-04,0.400000,0.400000,0.400000,0.564627,5.684522e-01
3406,179,Foreign Minister Datuk Seri Mohamad Hasan said...,4.000000e-01,0.400000,0.599098,0.400000,0.564627,9.484909e-01
3407,179,He said he was unclear about the real purpose ...,4.000000e-01,0.400000,0.400000,0.400000,0.400000,5.684522e-01
3408,179,"""I don't know the purpose because the Strait o...",4.000000e-01,0.400000,0.400000,0.400000,0.400000,5.684522e-01
3409,179,"Previously, Iran had agreed to open the strait...",2.225611e-04,0.400000,0.400000,0.400000,0.400000,4.000000e-01
3410,179,"""(Blocking the route) will further worsen the ...",4.000000e-01,0.528446,0.659287,0.400000,0.639447,5.684522e-01
3411,179,"The world economy, agriculture and energy will...",4.000000e-01,0.528446,0.808536,0.463862,0.476901,7.771629e-01
3412,179,"""So I ask America to reconsider this action an...",2.225611e-04,0.400000,0.400000,0.400000,0.476901,5.684522e-01
3413,179,"Mohamad, who is also the Member of Parliament ...",3.553770e-07,0.027659,0.000349,0.212920,0.090816,2.537387e-07


In [32]:
chosen = view_df.iloc[6]
print(f": [text]: "+chosen.text)
for aspect, lf_list in aspect_lf_groups.items():
    print(f"\n  {aspect:20s} {chosen[aspect]}")
    print(f"  {'-' * 40}")
    for lf in lf_list:
        result = lf(chosen)
        label  = "PRESENT" if result == 1 else "ABSTAIN" if result == -1 else "ABSENT"
        print(f"    {lf.name:40s}  →  {label}")

print("\n" + "=" * 60)

: [text]: "(Blocking the route) will further worsen the situation of energy supply throughout the world...

  Monetary_Financial   0.5
  ----------------------------------------
    lf_is_not_economic_news                   →  ABSTAIN
    lf_monetary_policy_nlp_matcher            →  ABSTAIN
    lf_monetary_policy_semantic               →  ABSTAIN
    lf_financial_markets_nlp_matcher          →  ABSTAIN
    lf_financial_markets_semantic             →  ABSTAIN
    lf_banking_credit_nlp_matcher             →  ABSTAIN
    lf_banking_credit_semantic                →  ABSTAIN
    lf_central_banks_nlp_matcher              →  ABSTAIN
    lf_central_banks_nlp                      →  ABSTAIN
    lf_rate_actions_nlp                       →  ABSTAIN
    lf_monetary_policy_inflation_exclusion    →  ABSTAIN
    lf_monetary_exclusion_clash               →  ABSTAIN

  Inflation_Prices     0.0
  ----------------------------------------
    lf_is_not_economic_news                   →  ABSTAIN
    lf_inf

In [14]:
embedding = embedder.encode(chosen.text, convert_to_tensor=True)
print(f'Similarity between:\n{chosen.text} and:\n')
for anchor in ANCHORS['energy_commodities']:
    anchor_embedding = embedder.encode(anchor, convert_to_tensor=True)
    similarity = util.cos_sim(embedding,anchor_embedding)
    print(f"{anchor}\n{similarity.item():.4f}")



NameError: name 'chosen' is not defined

In [18]:
# 1. INPUT YOUR OWN STRING HERE TO TEST
custom_text = "Slow rise in staple food prices"

# ==============================================================================
# INTERACTIVE TESTER
# ==============================================================================
import pandas as pd

# Create a mock series with the minimum required structure your LFs look for
mock_row = pd.Series({
    'text': custom_text,
    'title': '',                  # In case your LFs read the headline/title
    'is_headline': False,         # In case your LFs toggle logic based on placement
    'sentence_idx': 0
})

print(f"🔬 Testing Custom String:")
print(f"   [text]: \"{mock_row.text}\"\n")
print(f"{'Aspect':<25} | {'Labeling Function':<40} → Value")
print("-" * 75)

# Iterate through your grouping dictionary to test every LF
for aspect, lf_list in aspect_lf_groups.items():
    aspect_header_printed = False

    for lf in lf_list:
        # Call the rule/LF on our custom text payload
        result = lf(mock_row)

        # Format output string
        label = "PRESENT" if result == 1 else "ABSTAIN" if result == -1 else "ABSENT"

        # Aesthetic layout: Print the aspect header only on its first evaluated rule
        if not aspect_header_printed:
            aspect_name = aspect
            aspect_header_printed = True
        else:
            aspect_name = ""

        print(f"{aspect_name:<25} |   {lf.name:<38} → {label}")

    print("-" * 75)

🔬 Testing Custom String:
   [text]: "Slow rise in staple food prices"

Aspect                    | Labeling Function                        → Value
---------------------------------------------------------------------------
Monetary_Financial        |   lf_is_not_economic_news                → ABSTAIN
                          |   lf_monetary_policy_nlp_matcher         → ABSTAIN
                          |   lf_monetary_policy_semantic            → ABSTAIN
                          |   lf_financial_markets_nlp_matcher       → ABSTAIN
                          |   lf_financial_markets_semantic          → ABSTAIN
                          |   lf_banking_credit_nlp_matcher          → ABSTAIN
                          |   lf_banking_credit_semantic             → ABSTAIN
                          |   lf_central_banks_nlp_matcher           → ABSTAIN
                          |   lf_central_banks_nlp                   → ABSTAIN
                          |   lf_rate_actions_nlp                

In [29]:
embedding = embedder.encode(custom_text, convert_to_tensor=True)
print(f'Similarity between:\n{custom_text} and:\n')
for anchor in ANCHORS['inflation']:
    anchor_embedding = embedder.encode(anchor, convert_to_tensor=True)
    similarity = util.cos_sim(embedding,anchor_embedding)
    print(f"{anchor}\n{similarity.item():.4f}")



Similarity between:
Slow rise in staple food prices and:

cost of living has gone up
0.5837
Vulnerable households require targeted financial buffers to mitigate the direct economic impact of rising consumer prices and escalating cost of living pressures.
0.4680
Global supply chain shocks and shipping network disruptions are driving up raw material commodity input prices.
0.5920
As skyrocketing diesel costs ripple through the supply chain, economists warn that unmitigated fuel hikes are stoking broader inflation, leaving both commercial enterprises and household consumers vulnerable to a sharp increase in the cost of living.
0.5386
prices continue to climb year-on-year
0.5404
Inflation slowed considerably compared with the previous quarter.
0.5064
Price growth moderated as supply conditions improved.
0.7020
Consumer prices remained largely stable throughout the reporting period.
0.4585
Deflationary pressures emerged amid weakening demand.
0.5722
The pace of price increases has begun to 

In [17]:
# Swap out all-MiniLM for a contrastive baseline that ignores lexical dragging
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
embedding = model.encode(custom_text, convert_to_tensor=True)
print(f'Similarity between:\n{custom_text} and:\n')
for anchor in ANCHORS['inflation']:
    anchor_embedding = model.encode(anchor, convert_to_tensor=True)
    similarity = util.cos_sim(embedding,anchor_embedding)
    print(f"{anchor}\n{similarity.item():.4f}")



Similarity between:
Slow rise in staple food prices and:

cost of living has gone up
0.6519
Vulnerable households require targeted financial buffers to mitigate the direct economic impact of rising consumer prices and escalating cost of living pressures.
0.5870
Global supply chain shocks and shipping network disruptions are driving up raw material commodity input prices.
0.6323
As skyrocketing diesel costs ripple through the supply chain, economists warn that unmitigated fuel hikes are stoking broader inflation, leaving both commercial enterprises and household consumers vulnerable to a sharp increase in the cost of living.
0.6199
prices continue to climb year-on-year
0.6894
Inflation slowed considerably compared with the previous quarter.
0.6422
Price growth moderated as supply conditions improved.
0.6588
Consumer prices remained largely stable throughout the reporting period.
0.5569
Deflationary pressures emerged amid weakening demand.
0.6138
The pace of price increases has begun to 

In [ ]:
lf_fiscal_policy_keywords(chosen)

1